# Ejercicios 7, 8 y 9: indicadores, incorporacion de 2025 y evolucion
Las tablas agregadas se construyen con `scripts/build_indicadores.py` (`sql/07_indicadores.sql`) y el tablero de Metabase con `scripts/metabase_dashboard.py`. Documentacion: `docs/07_indicadores_metabase.md`, `docs/08_incorporacion_2025.md`, `docs/09_evolucion_2024_2026.md`.

In [1]:
import os, sys, subprocess, duckdb
sys.path.insert(0, '/workspace/scripts')
from run_sql import ejecutar
DB = '/workspace/data/processed/indicadores.duckdb'

## Ejercicio 7: construir las tablas de indicadores
Lee todos los anios descargados (~1-2 min). Si Metabase ya tenia abierta una version anterior, reinicielo despues (`docker compose restart metabase`). Se omite si la base ya existe; borre el archivo para reconstruirla.

In [2]:
if not os.path.exists(DB):
    subprocess.run([sys.executable, '/workspace/scripts/build_indicadores.py'], check=True)
con = duckdb.connect(DB, read_only=True)
con.sql('SELECT anio, servicio, count(*) AS meses, sum(registros) AS registros, sum(viajes) AS viajes_validos '
        'FROM ind_mensual GROUP BY ALL ORDER BY ALL').df()

,anio,servicio,meses,registros,viajes_validos
0,2024,green,12,660218.0,614513.0
1,2024,yellow,12,41169720.0,39563966.0
2,2025,green,12,591375.0,553441.0
3,2025,yellow,12,48722602.0,44021434.0
4,2026,green,8,337114.0,319186.0
5,2026,yellow,8,29703355.0,28145938.0


## Ejercicio 7: resultado de cada tarjeta del tablero
Las mismas consultas que usa Metabase (bloques `-- @tarjeta` de `sql/07_indicadores.sql`).

In [3]:
from build_indicadores import leer_bloques
for b in leer_bloques('/workspace/sql/07_indicadores.sql'):
    if b['tipo'] == 'tarjeta':
        print()
        print(f">> {b['id']} [{b['visualizacion']}] {b['titulo']}")
        print(con.execute(b['sql']).df().head(12).to_string(index=False))


>> K1 [scalar] Viajes validos (todos los anios, millones)
 viajes_millones
          113.22

>> K2 [scalar] Ingresos totales (millones USD)
 ingresos_millones_usd
                3284.9

>> K3 [scalar] Ticket medio (USD)
 ticket_medio_usd
            29.01

>> K4 [scalar] Participacion del taxi verde (% de viajes)
 pct_green
      1.31

>> I1 [line] P1. Demanda: viajes por mes, comparando anios (miles)
 mes anio  viajes_miles
   1 2024        2912.1
   1 2025        3286.8
   1 2026        3544.1
   2 2024        2941.4
   2 2025        3339.4
   2 2026        3236.5
   3 2024        3481.8
   3 2025        3861.8
   3 2026        3793.5
   4 2024        3454.9
   4 2025        3707.3
   4 2026        3705.4

>> I2 [bar] P2. Ingresos mensuales por servicio (millones USD)
   periodo servicio  ingresos_millones_usd
2024-01-01    green                   1.17
2024-01-01   yellow                  78.07
2024-02-01    green                   1.12
2024-02-01   yellow                  78.65
20

## Ejercicio 9: evolucion 2024 - 2025 - 2026
`sql/09_evolucion.sql` (comparaciones anuales con los mismos meses, ene-ago).

In [4]:
ejecutar('/workspace/sql/09_evolucion.sql', con)


>> Q1 (9.1) Resumen anual comparable (ene-ago), yellow + green
 anio  registros  viajes_validos  ingresos_millones_usd  ticket_medio_usd  tarifa_por_milla_usd  distancia_media_mi  velocidad_mph  pct_aeropuerto  pct_descartados
 2024 26831594.0      25833447.0                  728.9             28.22                  5.69                3.42          12.52           10.03             3.72
 2025 31954356.0      28968325.0                  818.3             28.25                  5.65                3.46          12.57            8.79             9.34
 2026 30040469.0      28465124.0                  857.5             30.13                  6.01                3.53          11.94            8.05             5.24

>> Q2 (9.2) Variacion interanual (ene-ago) por servicio, en %
 anio servicio     viajes  var_viajes_pct  var_ingresos_pct  var_ticket_pct
 2024    green   412620.0             NaN               NaN             NaN
 2025    green   369466.0           -10.5              -6.4      

 anio servicio  pct_viajes_con_cargo  recaudado_millones_usd  ticket_medio_usd
 2024    green                  0.00                     NaN             23.77
 2025    green                  9.88                    0.03             24.84
 2026    green                  8.58                    0.02             25.42
 2024   yellow                  0.00                     NaN             28.29
 2025   yellow                 73.10                   15.68             28.29
 2026   yellow                 72.51                   15.31             30.18

>> Q5 (9.5) Velocidad media en dia laboral: hora pico (7-19 h) vs resto, por anio
 anio  mph_hora_pico  mph_resto  pct_viajes_hora_pico
 2024          10.68      15.19                  71.8
 2025          10.48      15.15                  70.3
 2026          10.25      14.75                  69.3

>> Q6 (9.6) Mezcla de metodos de pago por anio (ene-ago, % de viajes)
        metodo_pago  pct_2024  pct_2025  pct_2026
            Tarjeta     73.